## Hybrid Bergman and LSTM residual-corrector

In [ ]:
!pip install -r ../requirements.txt

In [ ]:
# --- Metrics evaluation cell (append to single-cell notebook) ---
# This cell computes glucose-weighted metrics (gRMSE, gMAE), RMSE/MAE/MAPE
# for the hybrid Bergman + LSTM residual-corrector implementation above.
# It rebuilds the training/test split per subject (same logic as the model cell)
# and aggregates final-step predictions across the SUBJECTS_TO_PLOT subjects.
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt
from collections import Counter
import matplotlib.pyplot as plt


# --- Glucose-weighted penalty functions (from your snippet) ---
def sigmoid(x, a, epsilon):
xi = (2 / epsilon) * (x - a - (epsilon / 2))
if x <= a:
return 0
elif a < x <= a + (epsilon / 2):
return -0.5 * xi ** 4 - xi ** 3 + xi + 0.5
elif a + (epsilon / 2) < x <= a + epsilon:
return 0.5 * xi ** 4 - xi ** 3 + xi + 0.5
else:
return 1




def sigmoid_hat(x, a, epsilon):
xi_hat = - (2 / epsilon) * (x - a + (epsilon / 2))
if x <= a - epsilon:
return 1
elif a - epsilon < x <= a - (epsilon / 2):
return 0.5 * xi_hat ** 4 - xi_hat ** 3 + xi_hat + 0.5
elif a - (epsilon / 2) < x <= a:
return -0.5 * xi_hat ** 4 - xi_hat ** 3 + xi_hat + 0.5
else:
return 0




def penalty(g, g_hat):
alpha_L = 1.5; alpha_H = 1
beta_L = 30; beta_H = 100
gamma_L = 10; gamma_H = 20
T_L = 85; T_H = 155
sigma_T_L = sigmoid_hat(g, T_L, beta_L)
sigma_gamma_L = sigmoid(g_hat, g, gamma_L)
sigma_T_H = sigmoid(g, T_H, beta_H)
sigma_gamma_H = sigmoid_hat(g_hat, g, gamma_H)
return (1 + alpha_L * sigma_T_L * sigma_gamma_L + alpha_H * sigma_T_H * sigma_gamma_H)




def glucose_rmse(y_true, y_pred):
y_true = np.asarray(y_true, dtype=float)
y_pred = np.asarray(y_pred, dtype=float)
mask = np.isfinite(y_true) & np.isfinite(y_pred)
y_true, y_pred = y_true[mask], y_pred[mask]
penalties = np.array([penalty(g, g_hat) for g, g_hat in zip(y_true, y_pred)])
mse = np.nanmean(((y_true - y_pred) ** 2) * penalties)
return np.sqrt(mse)




def glucose_mae(y_true, y_pred):
y_true = np.asarray(y_true, dtype=float)